# 18: Export the website models to Hugging Face

Packages the detectors used by the website and uploads them to the model repository `tanish-237/ai-image-forensics-models`:

- `m1.pt`: ResNet-50 from E8 (half precision)
- `m2_lr.npz`: the logistic-regression weights on CLIP features from E8 (re-fitted from the saved features; the fit is deterministic)
- `attributor.pt`: the seven-class attribution model from E4
- `config.json`: which detector gives the verdict, names shown on the page

You need a Hugging Face **write** token (huggingface.co, Settings, Access Tokens, New token, type *Write*). It is typed into a hidden prompt and not saved in the notebook. No GPU needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import HfApi
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

ROOT = Path("/content/drive/MyDrive/minor-project")
DATA = ROOT / "processed_openfake_train_full"
OUT = Path("/content/website_models")
OUT.mkdir(exist_ok=True)

## Settings

In [ ]:
VARIANT = "full_aug"     # "full_aug" (full data + augmentation) or "full" (full data only), chosen from the E8 results
MAIN = "M2"              # detector that gives the verdict: "M1", "M2" or "mean"
REPO = "tanish-237/ai-image-forensics-models"
PRIVATE = False

## Package the models

In [ ]:
def save_half(src, dst):
    state = torch.load(src, map_location="cpu")
    torch.save({k: v.half() if v.is_floating_point() else v for k, v in state.items()}, dst)


save_half(ROOT / "checkpoints" / ("e8_m1_full_aug.pt" if VARIANT == "full_aug" else "e8_m1_full.pt"), OUT / "m1.pt")
save_half(ROOT / "checkpoints" / "e4_a2_resnet50.pt", OUT / "attributor.pt")

meta = pd.read_csv(DATA / "meta.csv")
y = (meta["label"] == "fake").astype(int).to_numpy()
train, val = (meta["split"] == "train").to_numpy(), (meta["split"] == "val").to_numpy()
X, Y = np.load(DATA / "clip_train.npy"), y[train]
if VARIANT == "full_aug":
    X = np.concatenate([X, np.load(DATA / "clip_train_aug1.npy"), np.load(DATA / "clip_train_aug2.npy")])
    Y = np.concatenate([Y] * 3)
f_val = np.load(DATA / "clip_val.npy")
search = [(log_loss(y[val], LogisticRegression(C=C, max_iter=5000).fit(X, Y).predict_proba(f_val)[:, 1]), C) for C in [0.1, 1, 10, 100]]
C = min(search)[1]
clf = LogisticRegression(C=C, max_iter=5000).fit(X, Y)
np.savez(OUT / "m2_lr.npz", coef=clf.coef_.astype(np.float32), intercept=np.array(clf.intercept_[0], dtype=np.float32))
print("M2 regularisation C =", C)

label = "trained on recent generators" + (", with augmentation" if VARIANT == "full_aug" else "")
config = {
    "main": MAIN,
    "clip_name": "openai/clip-vit-large-patch14",
    "sources": ["Real", "ADM", "GLIDE", "Midjourney", "SD15", "VQDM", "Wukong"],
    "names": {"M1": f"ResNet-50 ({label})", "M2": f"CLIP + linear classifier ({label})"},
    "description": f"E8 detectors ({VARIANT}); verdict from {MAIN}",
}
(OUT / "config.json").write_text(json.dumps(config, indent=2))
(OUT / "README.md").write_text(
    "---\nlicense: cc-by-nc-4.0\n---\n\n# AI Image Forensics models\n\n"
    "Detectors used by the website of our B.Tech minor project (MNNIT Allahabad, 2026). "
    "Trained on part of the OpenFake training split (CC BY-NC 4.0); the attribution model on Tiny GenImage (CC BY-NC-SA 4.0). "
    "For non-commercial research and teaching only.\n"
)
for f in sorted(OUT.iterdir()):
    print(f"{f.name}: {f.stat().st_size / 1e6:.1f} MB")

## Upload

In [ ]:
api = HfApi(token=getpass("Hugging Face write token: "))
api.create_repo(REPO, repo_type="model", private=PRIVATE, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=REPO, repo_type="model", commit_message="Website models")
print(f"Uploaded to https://huggingface.co/{REPO}")